In [5]:
from dotenv import load_dotenv
import os
load_dotenv()
gemini_key = os.getenv('Gemini_Api_Key')
if gemini_key is not None:
    print("Gemini key is loaded")

Gemini key is loaded


In [6]:
#sample input

In [8]:
patient_symptons = ['Fever' , 'Headache' , 'Soreness']

In [11]:
patient_condition = {
    "Age": 67,
    "Mobility Issues": "Requires cane for walking; limited stamina when standing for extended periods",
    "Known Allergies": ["Penicillin", "Latex"],
    "Chronic Conditions": ["Type 2 Diabetes", "Mild Hypertension"],
    "Recent Surgeries": "Hip replacement (8 months ago)",
    "Immunization Status": "Up-to-date, including flu and COVID-19 boosters",
    "Other Notes": "Reports occasional dizziness and sensitivity to cold weather"
}
patient_condition

{'Age': 67,
 'Mobility Issues': 'Requires cane for walking; limited stamina when standing for extended periods',
 'Known Allergies': ['Penicillin', 'Latex'],
 'Chronic Conditions': ['Type 2 Diabetes', 'Mild Hypertension'],
 'Recent Surgeries': 'Hip replacement (8 months ago)',
 'Immunization Status': 'Up-to-date, including flu and COVID-19 boosters',
 'Other Notes': 'Reports occasional dizziness and sensitivity to cold weather'}

In [34]:
geographic_location = {
    "Country": "United States",
    "State": "Kentucky",
    "City": "Highland Heights",
    "Zip Code": "41076",
    "Coordinates": {
        "Latitude": 39.1311,
        "Longitude": -84.5083
    },
    "Preferred Mode of Care": "In person"
}

In [14]:
#The first thing we would calculate is something like nearest health care centers or hospitals based on location

In [18]:
from geopy.distance import geodesic
import pandas as pd

In [21]:
# Sample list of hospitals with coordinates (for demo purposes)
hospital_data = [
    {"Name": "University of Cincinnati Medical Center", "Latitude": 39.1412, "Longitude": -84.5059},
    {"Name": "Cincinnati VA Medical Center", "Latitude": 39.1398, "Longitude": -84.5039},
    {"Name": "Christ Hospital", "Latitude": 39.1231, "Longitude": -84.5070},
    {"Name": "Cleveland Clinic", "Latitude": 41.5033, "Longitude": -81.6200},
    {"Name": "Mount Carmel East", "Latitude": 39.9828, "Longitude": -82.8291}
]
#make a dataframe of this, in reality this will I think be one dataframe per state, consisteing of all health care and hospital services in that area
df_hospitals = pd.DataFrame(hospital_data)


,Name,Latitude,Longitude
0,University of Cincinnati Medical Center,39.1412,-84.5059
1,Cincinnati VA Medical Center,39.1398,-84.5039
2,Christ Hospital,39.1231,-84.5070
3,Cleveland Clinic,41.5033,-81.6200
4,Mount Carmel East,39.9828,-82.8291


In [35]:
print("df_hospitals can also include things like what services are offered, which department and nuimber of employees; this is bascially a very big table")

df_hospitals can also include things like what services are offered, which department and nuimber of employees; this is bascially a very big table


In [23]:
#One example of  us calculating distance from patient to each hospital
latitude = geographic_location["Coordinates"]["Latitude"]
longitude = geographic_location["Coordinates"]["Longitude"]
patient_coords = (latitude, longitude)
patient_coords

(39.1311, -84.5083)

In [25]:
#I can do many things in order to calculate the actual distance based on latitide and longitide, maybe subtract both and keep the one with the lowest difference
# KNN? Overkill
# I think there might be libraries that do that properly
#geodesic offers a miles bersion that extract miles from distance, which is nice. I a,m going to do that

In [27]:
distances = []
# Calculate distance for each hospital
for index, row in df_hospitals.iterrows():
    hospital_coords = (row["Latitude"], row["Longitude"])
    distance = geodesic(patient_coords, hospital_coords).miles
    distances.append(distance)
distances

[0.7085628278211792,
 0.6450317424197475,
 0.5562695371259393,
 223.71261059390406,
 107.20597664086802]

In [28]:
#The lowest distance is our hospital

In [29]:
df_hospitals["Distance_miles"] = distances

In [37]:
closest_index = df_hospitals["Distance_miles"].idxmin() #find the id of the minimum
closest_hospital = df_hospitals.loc[closest_index, "Name"] #look up the name of the minimum
print("The closest hospital is", closest_hospital)
#Actually, gemini is smart enough for now to choose a hoispital for us based on the values as well.

The closest hospital is Christ Hospital


In [38]:
#I think I can use Gemini to build a basic pipeline for me now

In [39]:
#For now I will just use the base model but I can tinker with it later and use some RAG for the customized treatment plan.
#As long as I dont get the treatement plan, I wont be able to fine tune the model.
#Plus base gemini is jsut as fine, can do most of the stuff